# 62장 · MCP 도구와 LangChain 에이전트 연결

LangChain 도구가 MCP 서버의 함수를 호출하려면 클라이언트 세션이 열려 있어야 합니다. 응답의 형식과 요청에 해당하는 값이 돌아왔는지를 따로 확인합니다.

61장의 MCP 서버에 LangChain 도구를 연결합니다. 주말 운영 시간 요청을 열린 세션으로 전달하고 응답의 요일·시간과 도구 메시지를 대조합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langchain 1.4.6, langchain-core 1.6.12, langsmith 0.14.11, mcp 1.30.0입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 61장의 결과 열기

같은 폴더에 내려받은 노트북을 여세요. 먼저 my-course/agent-61.json를 읽습니다. 파일이 없으면 시작 셀의 완성 예시로 연습할 수 있습니다.

In [ ]:
from pathlib import Path
import json
folder = Path.cwd()
if (folder.name == "daily" or (folder.name.startswith("chapter") and folder.name[7:].isdigit())):
    folder = folder.parent
work = folder / "my-course"
work.mkdir(exist_ok=True)
source_file = work / "agent-61.json"
if source_file.exists():
    previous = json.loads(source_file.read_text(encoding="utf-8"))
else:
    previous = {'record': {'owner': 'alice', 'id': 'loan-01', 'text': '대출 기간은 14일입니다.'},
     'tool': 'lookup_loan',
     'args': {'doc_id': 'loan-01'},
     'reply': {'found': True, 'text': '대출 기간은 14일입니다.', 'style': 'short'}}
print("내 파일" if source_file.exists() else "완성 예시", previous["record"]["text"])

완성 예시 대출 기간은 14일입니다.


## 클라이언트 세션과 도구 어댑터

제공 셀에는 두 가지 코드가 있습니다. 위쪽은 서버의 시간표이고, 다음 셀의 어댑터는 LangChain 도구의 day 인자를 MCP 요청으로 바꿉니다. 메모리 전송이므로 원격 서버 배포나 인증 설정 없이 왕복을 확인할 수 있습니다.


### 셀 01 · FastMCP 도구 서버 준비

같은 서버, 같은 두 요일, 같은 자료 버전을 유지합니다.


In [1]:
from typing import Literal
from mcp.server.fastmcp import FastMCP
from mcp.shared.memory import create_connected_server_and_client_session

server = FastMCP("course-library-v1", log_level="ERROR")
hours = {"weekday": "09:00~18:00", "weekend": "10:00~16:00"}

@server.tool()
def get_library_hours(day: Literal["weekday", "weekend"]) -> dict[str, str]:
    """가상 도서관의 평일 또는 주말 운영 시간을 조회합니다."""
    return {"day": day, "hours": hours[day], "source": "수업용 가상 도서관"}

지금은 도구를 등록했습니다. 다음 셀에서 연결을 열 때까지 요청 횟수는 늘지 않습니다.


### 셀 02 · LangChain 도구에서 MCP 호출

ainvoke로 호출한 도구가 세션을 통해 받은 운영 시간만 돌려주게 합니다.

에이전트에는 lookup이라는 도구가 보입니다. lookup이 받은 day를 session.call_tool로 넘길 때 서버에는 get_library_hours라는 이름을 보냅니다. 서버 결과의 시간이 도구 메시지 내용으로 돌아오는 지점을 찾아보세요.


In [2]:
from langchain_core.tools import tool

def make_lookup(session):
    @tool
    async def lookup(day: Literal["weekday", "weekend"]) -> str:
        """가상 도서관의 운영 시간을 MCP로 조회한다."""
        reply = await session.call_tool("get_library_hours", {"day": day})
        if reply.isError:
            raise ValueError("remote_tool_error")
        data = reply.structuredContent
        if data["day"] != day:
            raise ValueError("result_day_mismatch")
        return data["hours"]
    return lookup

async with create_connected_server_and_client_session(server) as session:
    lookup = make_lookup(session)
    result = await lookup.ainvoke({"day": "weekend"})
print(result)

10:00~16:00


lookup은 생성될 때 전달받은 session으로 get_library_hours를 호출합니다. 따라서 에이전트의 실행까지 async with 블록 안에서 마쳐야 같은 연결로 결과를 받을 수 있습니다.

주말 시간 문자열이 돌아왔어도 어느 경로에서 얻었는지 구분해야 합니다. 여기서는 LangChain 쪽 도구 호출이 MCP 서버의 함수를 거칩니다. 연결 코드를 수정한 뒤에는 사람이 읽는 시간뿐 아니라 요청한 요일과 응답의 요일도 대조합니다.


### LangChain 도구가 호출하는 MCP 도구

<p>61장의 get_library_hours를 호출하는 쪽은 이제 LangChain 도구 lookup입니다. 두 이름 사이에서 day가 그대로 전달되는지 보세요.</p>

<h3>셀 04 · 평일 운영 시간이 도구 메시지에 담기는 과정</h3>

<p>실제 모델이 선택한 lookup의 인자, MCP 서버가 돌려준 시간, 최종 답변을 차례로 대조하세요. 실행 시 생성된 호출 ID로 요청과 결과를 연결합니다.</p>

<article><small>STEP 01 · 모델</small><h4>평일 인자의 도구 요청</h4>
<p>실제 모델이 선택한 lookup의 인자, MCP 서버가 돌려준 시간, 최종 답변을 차례로 대조하세요. 실행 시 생성된 호출 ID로 요청과 결과를 연결합니다.</p><pre>lookup(day="weekday") · id=call_qUxLpAH8lHUWDNL4WinDU43n</pre></article>

<article><small>STEP 02 · 어댑터</small><h4>MCP 세션의 호출 전달</h4>
<p>lookup은 현재 열린 세션에서 서버 도구를 부릅니다.</p><pre>session.call_tool("get_library_hours", {"day": "weekday"})</pre></article>

<article><small>STEP 03 · 서버</small><h4>서버의 시간표 조회</h4>
<p>서버가 요청 요일과 시간을 돌려줍니다. 어댑터가 요일을 확인하고 hours를 반환합니다.</p><pre>day = weekday
hours = 09:00~18:00</pre></article>

<article><small>STEP 04 · 프레임워크</small><h4>입력 요일과 반환값 대조</h4>
<p>도구가 돌려준 시간이 같은 ID의 도구 메시지 본문에 들어갑니다.</p><pre>ToolMessage · tool_call_id=call_qUxLpAH8lHUWDNL4WinDU43n
content = 09:00~18:00</pre></article>

<article><small>STEP 05 · 셀 04</small><h4>ToolMessage로 결과 반환</h4>
<p>실제 모델이 선택한 lookup의 인자, MCP 서버가 돌려준 시간, 최종 답변을 차례로 대조하세요. 실행 시 생성된 호출 ID로 요청과 결과를 연결합니다.</p><pre>call_qUxLpAH8lHUWDNL4WinDU43n 09:00~18:00</pre></article>


### 응답의 어느 값을 모델에 넘길까?

<p>MCP는 day와 hours 등이 담긴 결과를 돌려줍니다. lookup은 reply.isError를 먼저 확인하고 반환된 day가 요청과 일치할 때에만 hours 문자열을 넘깁니다.</p>


### 열린 세션의 수명

<p>make_lookup(session)은 도구가 쓸 세션을 기억합니다. 실제 lookup.ainvoke도 async with 블록 안에서 실행해야 합니다. 블록이 끝나면 연결 수명도 끝납니다.</p>


## 구조화된 도구 결과의 값 검증

운영 시간이 문자열인지 확인하는 것만으로 어느 날의 시간인지 보장할 수 없습니다. 받은 요일과 요청한 요일을 같이 검사합니다. 시간 초과는 이 검사보다 앞서 통신 단계에서 처리해야 합니다.


### 과제 03 · 요청 인자와 반환값 대조

주말 요청에 돌아온 응답이 평일입니다. 요일 불일치를 찾아내는 조건을 작성하세요.


In [3]:
received = {'day': 'weekday', 'hours': '09:00~18:00'}
requested = 'weekend'
# 받은 요일이 요청과 같은지 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>received = {&#x27;day&#x27;: &#x27;weekday&#x27;, &#x27;hours&#x27;: &#x27;09:00~18:00&#x27;}
requested = &#x27;weekend&#x27;
print(received[&#x27;day&#x27;] == requested)</code></pre>
<p>실행 결과</p><pre><code>False</code></pre>
<p>요일 비교 결과는 False입니다. 서버가 반환한 값 자체는 ‘09:00~18:00’으로 정상처럼 보이지만 주말 요청에는 쓸 수 없습니다. make_lookup에서 이 경우에 해당하는 오류 이름은 result_day_mismatch입니다.</p>
</details>


### 정상처럼 보이는 다른 요일의 결과

<p>받은 hours가 그럴듯해도 requested가 weekend이고 받은 day가 weekday이면 해당 요청의 답으로 쓸 수 없습니다. 이 경우 어댑터는 result_day_mismatch로 중단합니다.</p>


## create_agent와 ToolMessage의 결과 전달

실제 모델이 선택한 lookup의 인자, MCP 서버가 돌려준 시간, 최종 답변을 차례로 대조하세요. 실행 시 생성된 호출 ID로 요청과 결과를 연결합니다.

### 셀 04 · ToolMessage에서 도구 응답 추출

lookup에 등록한 실행 함수가 MCP를 한 번 부릅니다. 수업용 모델과 서버 모두 네트워크를 사용하지 않습니다.


In [ ]:
from langchain.agents import create_agent
from langsmith import tracing_context
import os
from langchain_openai import ChatOpenAI

# 수업 커널에 %pip install langchain-openai 를 실행하고 OPENAI_API_KEY를 설정하세요.
model = ChatOpenAI(model=os.getenv("OPENAI_MODEL") or "gpt-4o-mini", timeout=45, max_retries=1)
async with create_connected_server_and_client_session(server) as session:
    agent = create_agent(model, tools=[make_lookup(session)], system_prompt="운영 시간은 반드시 lookup으로 조회한 뒤 답하세요.")
    with tracing_context(enabled=False):
        state = await agent.ainvoke({"messages": [("user", "평일 운영 시간은?")]})
for message in state["messages"]:
    if message.type == "tool":
        print(message.tool_call_id, message.content)

call_qUxLpAH8lHUWDNL4WinDU43n 09:00~18:00


실제 모델이 선택한 lookup의 인자, MCP 서버가 돌려준 시간, 최종 답변을 차례로 대조하세요. 실행 시 생성된 호출 ID로 요청과 결과를 연결합니다.

#### MCP 결과가 모델의 대화에 들어오는 지점

<mark>어댑터가 받은 값과 ToolMessage에 담긴 값을 같은 호출 ID로 연결합니다.</mark> 서버가 돌려준 운영 시간이 정상이어도 요청한 요일과 다르면 다른 질문의 답입니다. 세션 안에서 값을 검사한 뒤 메시지의 call_qUxLpAH8lHUWDNL4WinDU43n과 요청을 대조하세요.

선수 개념 · 61장 FastMCP 도구 등록과 클라이언트 호출

#### 모델이 MCP 서버의 값을 받는 경로

```mermaid
flowchart TD
m["질문을 받은 실제 모델"] -->|실행 시 생성한 호출 ID| a["lookup 어댑터"] --> c["MCP 세션"] --> s["서버 시간표 도구"]
s --> v["응답 값 확인"] --> t["ToolMessage"] --> m
m -->|새 도구 요청 없음| e["최종 답변"]
```

실제 모델이 선택한 lookup의 인자, MCP 서버가 돌려준 시간, 최종 답변을 차례로 대조하세요. 실행 시 생성된 호출 ID로 요청과 결과를 연결합니다.

### 최종 문장과 도구의 반환값

<p>실제 모델이 선택한 lookup의 인자, MCP 서버가 돌려준 시간, 최종 답변을 차례로 대조하세요. 실행 시 생성된 호출 ID로 요청과 결과를 연결합니다.</p>


## 복습과 실습 점검

### 과제 05 · 서버가 다른 요일을 보냈을 때 거절하기

실제 모델이 선택한 lookup의 인자, MCP 서버가 돌려준 시간, 최종 답변을 차례로 대조하세요. 실행 시 생성된 호출 ID로 요청과 결과를 연결합니다.

In [ ]:
from types import SimpleNamespace

class WrongDaySession:
    async def call_tool(self, name, arguments):
        return SimpleNamespace(isError=False, structuredContent={"day": "weekday", "hours": "09:00~18:00"})

# make_lookup(WrongDaySession())에 주말을 요청하고 ValueError를 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>from types import SimpleNamespace

class WrongDaySession:
    async def call_tool(self, name, arguments):
        return SimpleNamespace(isError=False, structuredContent={&quot;day&quot;: &quot;weekday&quot;, &quot;hours&quot;: &quot;09:00~18:00&quot;})

lookup_wrong_day = make_lookup(WrongDaySession())
try:
    await lookup_wrong_day.ainvoke({&quot;day&quot;: &quot;weekend&quot;})
except ValueError as error:
    print(error)</code></pre>
<p>실행 결과</p><pre><code>result_day_mismatch</code></pre>
<p>응답에는 정상적인 시간 문자열이 들어 있지만 요청은 주말이고 응답은 평일입니다. 이 검사를 빼면 에이전트는 평일 시간을 주말 답으로 받습니다. 이 과제는 어댑터의 오류 분기만 실행하며 MCP 연결은 앞선 셀에서 확인합니다.</p>
</details>


## 같은 MCP 도구를 HTTP 경유 에이전트에 연결하기

61장에서 stdio 서버를 실행했습니다. 여기서는 FastMCP가 제공하는 Streamable HTTP 앱과 클라이언트를 연결합니다. HTTPX의 ASGITransport를 사용해 요청을 같은 프로세스 안의 앱에 전달하므로 외부 주소나 열린 포트 없이 HTTP 경로를 시험할 수 있습니다.

### 셀 06 · HTTP의 /mcp 경로에서 도구 목록 읽기

앞에서 정의한 get_library_hours 함수를 HTTP 서버에 등록합니다. with_http_session은 클라이언트를 초기화한 뒤 인자로 받은 작업을 실행하고 세션을 닫습니다. 첫 작업은 서버에 공개된 도구 이름을 읽는 것입니다.

In [5]:
import httpx
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

async def with_http_session(action):
    http_server = FastMCP("library-http", stateless_http=True, json_response=True, log_level="ERROR")
    http_server.tool()(get_library_hours)
    http_app = http_server.streamable_http_app()
    transport = httpx.ASGITransport(app=http_app)
    async with http_server.session_manager.run():
        async with httpx.AsyncClient(transport=transport) as client:
            async with streamable_http_client("http://127.0.0.1:8000/mcp", http_client=client) as channels:
                async with ClientSession(channels[0], channels[1]) as session:
                    await session.initialize()
                    return await action(session)

async def names_from(session):
    return [item.name for item in (await session.list_tools()).tools]

print(await with_http_session(names_from))

['get_library_hours']


get_library_hours가 보이면 HTTP 경로로 목록 조회에 성공한 것입니다. 이 결과만으로는 운영 시간 응답이 사용자에게 돌아오는지 알 수 없습니다. 다음 과제에서는 같은 경로로 실제 도구를 호출합니다.

### 과제 07 · 에이전트의 도구 결과에서 호출 ID와 시간 찾기

실제 모델이 선택한 lookup의 인자, MCP 서버가 돌려준 시간, 최종 답변을 차례로 대조하세요. 실행 시 생성된 호출 ID로 요청과 결과를 연결합니다.

In [6]:
# async def ask_over_http(session): ...
# await with_http_session(ask_over_http)로 실행하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>async def ask_over_http(session):
    service_agent = create_agent(model, tools=[make_lookup(session)], system_prompt=&quot;시간은 lookup으로 조회하세요.&quot;)
    with tracing_context(enabled=False):
        result = await service_agent.ainvoke({&quot;messages&quot;: [(&quot;user&quot;, &quot;주말에는 언제 여나요?&quot;)]})
    reply = next(m for m in result[&quot;messages&quot;] if m.type == &quot;tool&quot;)
    return reply.tool_call_id, reply.content

print(await with_http_session(ask_over_http))</code></pre>
<p>실행 결과</p><pre><code>(&#x27;call_0SY2QCqfLmJBzRE9cxGOtc01&#x27;, &#x27;10:00~16:00&#x27;)</code></pre>
<p>실제 모델이 선택한 lookup의 인자, MCP 서버가 돌려준 시간, 최종 답변을 차례로 대조하세요. 실행 시 생성된 호출 ID로 요청과 결과를 연결합니다.</p>
</details>

<mark>목록에서 본 서버 도구 이름과 모델이 요청한 도구 이름은 다를 수 있습니다.</mark> 이 어댑터에서 모델은 lookup을 요청하고 MCP 세션은 get_library_hours를 요청합니다. 연결 문제를 찾을 때는 두 이름을 각각의 등록 위치에서 확인하세요.

같은 MCP 어댑터를 실제 주소로 옮길 때는 네트워크 연결과 접근 권한을 별도로 점검합니다.

## 비동기 LangChain 도구에서 MCP 결과 읽기

61장이 남긴 이름과 인자를 에이전트가 호출할 수 있는 도구에 연결합니다. 이전 응답은 비교용으로 남기고 새 MCP 세션에서 조회를 다시 실행하세요.

In [ ]:
from langchain_core.tools import tool
from mcp.server.fastmcp import FastMCP
from mcp.shared.memory import create_connected_server_and_client_session
record = previous["record"]
desk_server = FastMCP("loan-desk", log_level="ERROR")
@desk_server.tool()
def lookup_loan(doc_id: str) -> dict[str, str | bool]:
    """문서 번호로 규정을 조회한다."""
    return {"text": record["text"]} if doc_id == record["id"] else {}
tool_events = []
async with create_connected_server_and_client_session(desk_server) as desk_session:
    @tool
    async def read_rule(doc_id: str) -> str:
        """문서 번호로 MCP 서버의 대출 규정 본문을 읽는다."""
        result = await desk_session.call_tool(previous["tool"], {"doc_id": doc_id})
        if result.isError:
            raise RuntimeError("MCP 조회 실패")
        tool_events.append({"name": previous["tool"], "args": {"doc_id": doc_id}})
        return result.structuredContent.get("text", "")
    linked_text = await read_rule.ainvoke(previous["args"])
print(linked_text, linked_text == previous["reply"]["text"])

대출 기간은 14일입니다. True


ainvoke가 MCP 응답에서 text만 꺼냅니다. 기본 자료에서는 61장의 원문과 같아 True가 나옵니다. 여기서는 에이전트 선택 이전에 어댑터를 직접 시험합니다.

### 직접 확인하기

61장 기록의 번호와 새 조회의 원문을 한 쌍으로 묶으세요. 인용을 만들 사람이 두 값을 각각 꺼낼 수 있어야 합니다.

In [ ]:
# args에서 문서 번호를, linked_text에서 조회 응답을 읽어 딕셔너리를 만드세요.

<details><summary>정답 코드와 실행 결과 보기</summary><pre><code>citation = {&quot;id&quot;: previous[&quot;args&quot;][&quot;doc_id&quot;], &quot;text&quot;: linked_text}
print(citation[&quot;id&quot;], citation[&quot;text&quot;])</code></pre><p>기본 자료로 실행한 결과</p><pre><code>loan-01 대출 기간은 14일입니다.</code></pre><p>도구 호출에 넣은 번호와 같은 실행의 원문을 묶습니다. 54장에서 바꾼 원문이라면 새 값이 나옵니다.</p></details>

### 63장에 넘길 결과 저장하기

63장은 이 근거를 서비스 응답의 sources에 넣습니다. 저장 셀을 다시 실행하면 같은 파일에 이번 결과를 기록합니다.

In [ ]:
product = {"record": record, "evidence": {"id": previous["args"]["doc_id"], "text": linked_text}, "events": tool_events}
target = work / "agent-62.json"
target.write_text(json.dumps(product, ensure_ascii=False, indent=2), encoding="utf-8")
print(target.name)

agent-62.json
